# Molecules Journey - Active Learning

Simple tutorial on `MoleculesJourney`.

In [ ]:
from pathlib import Path

from scm.moliterate import PropertyInfo
from scm.moliterate.analysis import PairwiseDatasetMetrics
from scm.moliterate.core.chem_data_entry import ChemDataEntry
from scm.moliterate.interfaces import ASEMolData as ASEDatabase
from scm.plams import from_smiles

from scm.active_learning import ActiveLearningLoop, MoleculesJourney
from scm.active_learning.checker_getters.checkers import AMSTrajChecker
from scm.active_learning.checker_getters.getters import AMSTrajGetter
from scm.active_learning.engines import AMSEngine
from scm.active_learning.loop import StoppableCounter
from scm.active_learning.mlip.params_trainer import ParAMSTrainer
from scm.active_learning.tasks import AMSGOIRTask, AMSMDTask, SPLabeller

In [2]:
smiles_list = ["CO", "CCO", "CC", "O"]
db_path = Path("molecules.db")
if db_path.exists():
    db_path.unlink()

molecules = ASEDatabase.create(db_path, available_properties=[])
molecules.add_systems(
    ChemDataEntry(system=from_smiles(smiles), metadata={"smiles": smiles})
    for smiles in smiles_list
)

4it [00:00, 578.88it/s]


In [3]:
md_settings = AMSMDTask.model_construct(nsteps=50, thermostat = "NHC", temperature = 300.0, samplingfreq = 2).settings
md_settings2 =  AMSMDTask.model_construct(nsteps=50, thermostat = "NHC", temperature = 700.0, samplingfreq = 2).settings
md_checker_getter = AMSTrajChecker() + AMSTrajGetter(data_selector_if_low_data=(None, None, 8))

In [4]:
go_settings = AMSGOIRTask.model_construct(normal_modes=False).settings
go_simple_checker_getter = AMSTrajChecker() + AMSTrajGetter(data_selector_if_low_data=(None, None, 3))
# go_checker_getter = go_simple_checker_getter >> (AMSIRCommitteeAgreementChecker() + NMSGetters(max_n_structures=5))

In [5]:
molecules_journey = MoleculesJourney(
    molecules=molecules,
    task_checker_getter={
        "md300": ("ams", md_settings.as_dict(), md_checker_getter),
        "md700": ("ams", md_settings2.as_dict(), md_checker_getter),
        "go": ("ams", go_settings.as_dict(), go_simple_checker_getter),
    },
    max_attempts_per_task={"md300":1, "md700":1, "go":2},
    batch_size=2,
)

In [6]:
import sys

ActiveLearningLoop.logging_config(
    clean_sinks="YES",
    new_sink=sys.stderr,
    level="INFO",
)

1

In [7]:
al = ActiveLearningLoop(
    start_engine=AMSEngine.Builder.UFF().build(),
    journey=molecules_journey,
    iterable_loop=StoppableCounter(stop=3),
    labeller=SPLabeller(properties=[
        PropertyInfo(name="energy", unit="eV"), 
        PropertyInfo(name="forces", unit="eV/Ang"),
    ]),
    labeller_engine=AMSEngine.Builder.UFF().build(), 
    # labeller_engine=AMSEngine.Builder.DFTB_GFN1().build(),
    accuracy_checker = PairwiseDatasetMetrics(settings=[
        PairwiseDatasetMetrics.PropMetricEv(property="energy", metric="mae", per_n_atoms=True, target=0.01)
    ]),
    mlip_trainer=ParAMSTrainer.Builder.TEST().set_committee(1).build(),
)

In [8]:
result_engine = al.run()

2026-04-21 15:24:48 | INFO     | AL Loop Folder: ALruns/20260421_152448
2026-04-21 15:24:48 | INFO     | Train Val created: ALruns/20260421_152448/train_validation_AL.db
2026-04-21 15:24:48 | INFO     | 
===== MoleculesJourney Scheduler =====
MoleculesJourney | NMolecules=4
task    checker           max_attempts
------  --------------  --------------
md300   AMSTrajChecker               1
md700   AMSTrajChecker               1
go      AMSTrajChecker               2
2026-04-21 15:24:48 | INFO     | ManualStopper: to stop the loop manually, run:
 echo "message" > /home/bene/Documents/work/ALIR/Code/active_learning_workspace/active_learning/tutorials/notebooks/ALruns/20260421_152448/STOP_ACTIVE_LEARNING
2026-04-21 15:24:48 | INFO     | ===================== iAL:00 =====================
/home/bene/Documents/work/ALIR/Code/active_learning_workspace/moliterate/src/scm/moliterate/interfaces/rkf_files.py:102: RKFDataWarning: MDHistory is not found. It has been removed self.sections=('History',

In [9]:
al.analysis.get_summary()

{'Reason': 'AL stopped: max AL iterations reached',
 'NSteps': 3,
 'Ninit': 0,
 'Nfinal': 27,
 'Tot[min]': 0}

In [10]:
import webbrowser

webbrowser.open(str(al.analysis.plot.save_to_pdf()))

ALruns/20260421_152448/analysis.pdf


True

In [11]:
print(molecules_journey.history_table())

=============================================== MoleculesJourney History ===============================================
  Idx    tot_pending    tot_inactivated    tot_succeeded    n_running    n_inactivated    n_succeeded  finished_msg    all_running_succeeded
-----  -------------  -----------------  ---------------  -----------  ---------------  -------------  --------------  -----------------------
    0              2                  0                0            2                0              0                  False
    1              2                  0                0            2                0              0                  False
=================================== Recent Failures ===================================
Molecule    MStatus    Task    TStatus      MaxAttempts  History
----------  ---------  ------  ---------  -------------  -------------------------------
M0000       Running    md300   Finished               1  AccuracyFailure-AccuracyFailure
M0000       Ru

In [12]:
# import shutil

# from scm.active_learning.callbacks import (
#     FolderManagerCallback,
# )

# shutil.rmtree(al.query_callbacks(FolderManagerCallback)[0].run_dir())